# Warning: Due to space constraints from running my tests I clear out the zip and unzipped folders. Ensure you have the zip stored in a separate location for additional operations.

# Note it was found that I could not easily dynamically pull the notebook name so if you update the notebook update the following code:

In [1]:
nb_name = "MAT-12-03-search-query-notebook"

In [2]:
# sudo apt install python3-dmidecode
#!pip install llama-cpp-python   --upgrade   --force-reinstall   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu126
#!pip3 install torch==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
!export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH

In [3]:
# Native
import concurrent.futures
from datetime import datetime
import hashlib
import os
from pathlib import Path
import shutil
import sys
import time
import zipfile

# Third Party
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm

/home/flaniganp/miniconda3/envs/my-python-buddy-notebook/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Get project base directory (one level up from current working directory)
base_dir = Path.cwd().parent

base_application_dir = base_dir / "base_application"

# Convert to absolute string path
base_application_dir = str(base_application_dir.resolve())

# Add to Python path if not already present
if base_application_dir not in sys.path:
    sys.path.append(base_application_dir)

print("base_application added to PATH:")
print(base_application_dir)

base_application added to PATH:
/home/flaniganp/Documents/my-python-buddy/base_application


In [5]:
from views_chat_utilities import build_summary_prompt, clean_and_build_summary, perform_brave_search_query

In [6]:
!django-admin startproject temp_project

CommandError: '/home/flaniganp/Documents/my-python-buddy/notebook/temp_project' already exists


In [7]:
import os
import getpass
import django
from django.conf import settings

# Set Django settings module
os.environ["DJANGO_SETTINGS_MODULE"] = "temp_project.temp_project.settings"

# Initialize Django
django.setup()

# Prompt user securely for Brave API key
BRAVE_API_KEY = getpass.getpass("Enter your Brave API key: ")

# Store it as an environment variable
os.environ["BRAVE_API_KEY"] = BRAVE_API_KEY

# Update Django settings dynamically
setattr(settings, "BRAVE_API_KEY", BRAVE_API_KEY)

# Confirm it’s now available
print("Brave API Key loaded successfully.")

Enter your Brave API key:  ········


Brave API Key loaded successfully.


In [8]:
RELEVANT_SECURITY_DOMAINS = {
    "boost": [
        # Manually found good sources
        "arjancodes.com",
        # Core analyzers and secure-coding authorities
        "bandit.readthedocs.io",
        "semgrep.dev",
        "mypy.readthedocs.io",
        "docs.python.org",
        "owasp.org",
        # Reliable security blogs / research
        "snyk.io",
        "securitylab.github.com",
    ],
    "discard": ["reddit.com", "stackoverflow.com", "github.com", "gitlab.com", "quora.com", "geeksforgeeks.org"],
}
# Update Django settings dynamically
setattr(settings, "RELEVANT_SECURITY_DOMAINS", RELEVANT_SECURITY_DOMAINS)

In [9]:
# Generates a cleaned text response using a Hugging Face model pipeline (non-llama specific).
def get_cleaned_summary_response_merged_model(llm_model, tokenizer, user_question):
    # Perform search query and retrieve top snippets
    citation_block, formatted_snippet_block = perform_brave_search_query(user_question)

    # Strong factual summarizer prompt (revised)
    max_new_tokens = 360
    summarize_max_words = max_new_tokens / 4

    system_prompt = build_summary_prompt(summarize_max_words, user_question, formatted_snippet_block)

    # Build conversation structure
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question},
    ]

    # Attempt to build a structured prompt (chat template if available)
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        bos = tokenizer.bos_token or ""
        prompt = f"{bos}[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_question} [/INST]"

    # Configure model for inference
    llm_model.eval()
    try:
        # Some models may not support gradient checkpointing disable
        llm_model.gradient_checkpointing_disable()  # noqa: B110
    except AttributeError:
        # Attribute may not exist on all model types and is safe to ignore
        pass
    llm_model.config.use_cache = True

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Generation settings
    max_new_tokens = 300
    generation_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        no_repeat_ngram_size=6,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False,
    )

    # print("SYSTEM PROMPT:\n", system_prompt[:500])
    # print("SNIPPET LENGTH:", len(formatted_snippet_block))

    # Create pipeline and generate
    text_pipe = pipeline(
        task="text-generation",
        model=llm_model,
        tokenizer=tokenizer,
        device_map="auto",
    )

    result = text_pipe(prompt, **generation_kwargs)

    # print("RESULT:", result)

    # Extract generated text and token counts
    generated_text = result[0]["generated_text"].strip()
    prompt_tokens = len(tokenizer.encode(prompt))
    max_tokens_used = max_new_tokens

    # Postprocess text (placeholder for cleaning or formatting)
    cleaned_text = clean_and_build_summary(generated_text, citation_block, formatted_snippet_block)

    return cleaned_text, prompt_tokens, max_tokens_used

In [10]:
# Compute SHA256 of a file
def sha256sum(file_path, block_size=65536):
    sha = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(block_size), b""):
            sha.update(chunk)
    return sha.hexdigest()

In [11]:
# Define paths
zip_path = os.path.join(base_dir, "models", "llama-2-7b-143k-codeAlpaca-2025-10-30_1326.zip")
hash_path = os.path.join(base_dir, "models", "llama-2-merged-7b-143k-codeAlpaca-2025-10-30_1326-hash.txt")
extract_dir = os.path.join(base_dir, "models", "llama-2-merged-7b-14k-codeAlpaca-2025-10-30_1143")

In [12]:
if os.path.exists(extract_dir):
    try:
        shutil.rmtree(extract_dir)
        print(f"Successfully deleted directory: {extract_dir}")
    except Exception as e:
        print(f"Error deleting {extract_dir}: {e}")
else:
    print(f"Directory not found: {extract_dir}.")

Directory not found: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-14k-codeAlpaca-2025-10-30_1143.


In [13]:
# Read expected hash
with open(hash_path, "r") as f:
    expected_hash = f.read().strip()

# Compute actual hash
actual_hash = sha256sum(zip_path)

# Compare
if actual_hash == expected_hash:
    print(f"Hash verified: {actual_hash}")
else:
    raise Exception((f"Hash mismatch!\nExpected: {expected_hash}\nFound: {actual_hash}"))

Hash verified: a7313874122da306d722fa8d77854a93cba398576f1a9b240830fbb6723e180b


In [14]:
if not os.path.exists(extract_dir):
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.infolist()

        # Define a worker that extracts one file at a time
        def extract_member(member):
            zf.extract(member, extract_dir)

        # Use ThreadPoolExecutor (I/O bound) instead of ProcessPoolExecutor
        # because ZipFile objects aren’t pickle-safe
        with concurrent.futures.ThreadPoolExecutor() as executor:
            list(
                tqdm(
                    executor.map(extract_member, members),
                    total=len(members),
                    desc="Extracting",
                )
            )
    print(f"Extracted to: {extract_dir}")
else:
    print(f"Directory already exists: {extract_dir}")

Extracting: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:20<00:00,  1.67s/it]

Extracted to: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-14k-codeAlpaca-2025-10-30_1143


In [15]:
# Remove archive (comment out if you want to keep it)
if os.path.exists(zip_path):
    try:
        os.remove(zip_path)
        print(f"Successfully deleted archive: {zip_path}")
    except Exception as e:
        print(f"Error deleting file {zip_path}: {e}")
else:
    print(f"File not found: {zip_path}")

Successfully deleted archive: /home/flaniganp/Documents/my-python-buddy/models/llama-2-7b-143k-codeAlpaca-2025-10-30_1326.zip


In [16]:
# Load model and tokenizer
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(extract_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(extract_dir)

# Confirm load
print(f"Model and tokenizer loaded from {extract_dir}")

Loading model...


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:05<00:00,  1.67s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


Model and tokenizer loaded from /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-14k-codeAlpaca-2025-10-30_1143


In [17]:
questions = [
    "What is a secure alternative to the python pickle module?",
    "What are the security implications for the subprocess module?",
    "How should I mitigate 'subprocess call with shell=True identified'?",
    "What is a secure hashing algorithm to use instead of md5?",
    "What are common hardcoded secret patterns detected by Dodgy in Python?",
]

In [18]:
# Setup output directory and file
base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
results_dir = base_dir / "test_results"
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
output_file = results_dir / f"{nb_name}_{timestamp}.txt"

execution_times = []  # store all durations

# Begin loop over questions
with open(output_file, "w", encoding="utf-8") as f:
    for i, user_question in enumerate(questions, start=1):
        print(f"\nQuestion {i}: {user_question}")
        f.write(f"\nQuestion {i}: {user_question}\n")

        # --- Measure time ---
        start_time = time.time()
        cleaned_response, prompt_tokens, max_new_tokens = get_cleaned_summary_response_merged_model(model, tokenizer, user_question)
        end_time = time.time()
        duration = end_time - start_time
        execution_times.append(duration)

        # --- Display and record results ---
        print(f"Cleaned Response:\n{cleaned_response}")
        print("-" * 88)

        f.write(f"Cleaned Response:\n{cleaned_response}\n")
        f.write(f"Prompt Tokens: {prompt_tokens}\n")
        f.write(f"Max New Tokens: {max_new_tokens}\n")
        f.write(f"Time Taken: {duration:.2f} seconds\n")
        f.write("Is result correct? If not, one sentence as to why:\n\n")
        f.write("-" * 88 + "\n")

    # Compute and record summary stats
    if execution_times:
        low_time = min(execution_times)
        high_time = max(execution_times)
        avg_time = sum(execution_times) / len(execution_times)

        summary = (
            f"\nExecution Time Summary:\n"
            f"Lowest Time:  {low_time:.2f} seconds\n"
            f"Highest Time: {high_time:.2f} seconds\n"
            f"Average Time: {avg_time:.2f} seconds\n"
        )

        print(summary)
        f.write(summary)

print(f"\nAll results recorded to: {output_file}")


Question 1: What is a secure alternative to the python pickle module?


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Cleaned Response:
A secure alternative to the python Pickle module is JSON.

It is a lightweight data interchange format that is easy for humans to read and write, and for machines to parse and generate.

It is a text format that uses human-readable characters, and it is based on a subset of the JavaScript Object Notation (JSON) data format.

JSON is a standard for data exchange between web servers and web browsers, and it is simple enough for humans to read and generate.

It is also fast, compact, and easy for machines to parse and produce.

Source(s):
  1.  Python Pickle Risks and Safer Serialization Alternatives | ArjanCodes;  (https://www.arjancodes.com/blog/python-pickle-module-security-risks-and-safer-alternatives/)
  2.  pickle — Python object serialization;  (https://docs.python.org/3/library/pickle.html)
  3.  Insecure Deserialization in Python | Semgrep;  (https://semgrep.dev/docs/learn/vulnerabilities/insecure-deserialization/python)
  4.  larch-pickle - Python Package Healt

Device set to use cuda:0


Cleaned Response:
The subprocess module in Python can be used to execute shell commands, but it's important to use it securely to avoid command injection vulnerabilities.

When not used securely, it can lead to arbitrary command execution, which is a serious security flaw.

Therefore, it's recommended to avoid using shell=True in cases where the command contains unsanitized input.

Additionally, it's important to sanitize any user-controlled inputs to prevent variable, glob patterns, and special shell features from being processed before the command is executed.

Source(s):
  1.  subprocess — Subprocess management;  (https://docs.python.org/3/library/subprocess.html)
  2.  Command injection in Python: examples and prevention | Snyk;  (https://snyk.io/blog/command-injection-python-prevention-examples/)
  3.  17.1. subprocess — Subprocess management — Python v2.7.3 documentation;  (https://docs.python.org/release/2.7.3/library/subprocess.html)
  4.  Command Injection in Python | Semgrep;

...

Cleaned Response:
The best way to mitigate'subprocessing call with shell=True' is to sanitize all inputs to prevent any potential shell injection attacks.

This can be done by using the `subprocess.check_call` function with the `shell=False` argument, which will execute the command directly without passing it through the shell.

Additionally, you should use the `subprocess.Popen` function instead of `subprocess.check-call`, which allows you to specify the shell argument as a string, allowing you to control the shell environment.

Here is an example of how you could modify your code to use these safer methods: ```python
import subprocess def check_call(command, shell=False): if shell: return subprocess.check_output(command, shell=True) else: return subprocessor.check_call(command, stdout=subprocess.PIPE, stderr=subprocess.PIPED, shell=False) def main(): print(check_call('ls -l', shell = True)) print(check.call('ls -l')) if __name__ == "__main__": main()
``` In this example, `check_call`

Device set to use cuda:0


Cleaned Response:
A secure hashing algorithm to replace MD5 is SHA-254 or SHA-384.

These are more secure than MD5 and are widely used in cryptographic applications.

Source(s):
  1.  What is an insecure hash? | Tutorial & examples | Snyk Learn;  (https://learn.snyk.io/lesson/insecure-hash/)
  2.  Python 3.14 documentation;  (https://docs.python.org/3/library/sha.html)
  3.  Password Storage - OWASP Cheat Sheet Series;  (https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html)
  4.  Find Weak Cryptographic Algorithms: Bye Bye MD5 | Testing for Weak Cryptography with Snyk | Snyk;  (https://snyk.io/blog/can-snyk-find-weak-cryptographic-algorithms-bye-bye-md5/)
  5.  hashlib — Secure hashes and message digests;  (https://docs.python.org/3/library/hashlib.html)
----------------------------------------------------------------------------------------

Question 5: What are common hardcoded secret patterns detected by Dodgy in Python?


Device set to use cuda:0


Cleaned Response:
The following are common hardcoded secret pattern detected by Dodgy: 1.

API keys and other credentials stored in plain text files or configuration files.
2.

Secrets stored in databases without encryption.
3.

Insufficient secretes rotation.
4.

Hardcoding secrets in code.
5.

Using hardcoded secrets in functions or methods.
6.

Storing secrets in environment variables.
7.

Using hardcoded secretes in configuration files.
8.

Storing secrets as part of the build process.
9.

Using hardcoded secreets in deployment scripts.
10.

Storing secrets on the server.

These patterns are often used by attackers to gain unauthorized access to systems and data.

It is important to detect and prevent these patterns in order to maintain the security of systems and data.

Source(s):
  1.  Hardcoded secrets | Tutorial and examples | Snyk Learn;  (https://learn.snyk.io/lesson/hardcoded-secrets/)
  2.  Want to avoid a data breach? Employ secrets detection - SEO | Snyk;  (https://snyk.i

In [19]:
# Cleanup extracted directory
try:
    shutil.rmtree(extract_dir)
    print(f"Successfully deleted directory: {extract_dir}")
except Exception as e:
    print(f"Error deleting {extract_dir}: {e}")

Successfully deleted directory: /home/flaniganp/Documents/my-python-buddy/models/llama-2-merged-7b-14k-codeAlpaca-2025-10-30_1143


In [20]:
# Django Cleanup

# Path to your temp project (adjust if needed)
temp_project_path = os.path.join(os.getcwd(), "temp_project")

# Remove the directory and all contents
if os.path.exists(temp_project_path):
    shutil.rmtree(temp_project_path)
    print(f"Removed: {temp_project_path}")
else:
    print("No temp_project folder found.")

Removed: /home/flaniganp/Documents/my-python-buddy/notebook/temp_project
